# S-train

Classifier training with between-class augmentation and threshold control.

In [ ]:
import os, datetime
import numpy as np
import tensorflow as tf

from nets.MobileNetV3_m import build_classifier
from utils.dataloader import make_dataset
from utils.utils_fit import focal_loss
from utils.utils_aug import mix
from utils.callbacks import ThresholdController, History, SaveWeights, cosine_schedule
from utils.utils import show_config

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

DATA        = 'data/rdtd_labeled/'
CHANNELS    = 3
MID_UNITS   = 128
DROPOUT     = 0.2
EPOCHS      = 100
BATCH       = 64
LR          = 1e-4
TRAIN_RATIO = 0.8
MIX         = 'interior'      # 'interior' | 'boundary' | None
SEED        = 42
# VAL_GROUPS = ['S4'] would hold out whole structures - recommended whenever a
# transfer/robustness claim is made; None keeps the random record-level split.
VAL_GROUPS  = None
OUT         = 'runs/' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

tf.random.set_seed(SEED)
np.random.seed(SEED)
for g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(g, True)


In [ ]:
train_ds, val_ds, classes = make_dataset(DATA, BATCH, TRAIN_RATIO,
                                        channels=CHANNELS, val_groups=VAL_GROUPS)
steps = int(train_ds.cardinality().numpy())
show_config(data=DATA, classes=classes, steps_per_epoch=steps, epochs=EPOCHS,
            batch=BATCH, lr=LR, mid_units=MID_UNITS, dropout=DROPOUT,
            mix=MIX, val_groups=VAL_GROUPS)


In [ ]:
model = build_classifier(CHANNELS, 1, MID_UNITS, DROPOUT)
model.summary()

optimizer = tf.keras.optimizers.Adam(
    cosine_schedule(LR, LR * 0.01, total_steps=steps * EPOCHS))

In [ ]:
os.makedirs(OUT, exist_ok=True)
ctrl = ThresholdController(val_ds)
history, saver = History(OUT), SaveWeights(OUT, period=10)
ctrl.set_model(model)
saver.set_model(model)
run_loss = tf.keras.metrics.Mean()

for epoch in range(EPOCHS):
    a_pos, a_neg = ctrl.weights()

    for images, labels in train_ds:
        if MIX:
            images, labels, _, _ = mix(images, labels, mode=MIX, training=True)
        with tf.GradientTape() as tape:
            preds = model(images, training=True)
            loss = focal_loss(labels, preds, a_pos, a_neg) + model.reg_loss()
        optimizer.apply_gradients(
            zip(tape.gradient(loss, model.trainable_variables),
                model.trainable_variables))
        run_loss.update_state(loss)

    # the epoch-mean loss feeds the logs (the old loop logged the last batch)
    ctrl_logs = {'loss': float(run_loss.result())}
    run_loss.reset_states()
    res = ctrl.on_epoch_end(epoch, ctrl_logs)
    if res is None:
        break

    logs = dict(ctrl_logs, val_loss=1 - res['f1'],
                val_f1=res['f1'], val_auc=res['auc'])
    history.on_epoch_end(epoch, logs)
    saver.on_epoch_end(epoch, logs)

    print(f"epoch {epoch+1:3d}  auc={res['auc']:.3f}  J={res['J']:.3f}  "
          f"tpr={res['tpr']:.3f}  tnr={res['tnr']:.3f}  thr={res['thr']:.3f}  "
          f"a=({a_pos:.2f}/{a_neg:.2f})")

    if ctrl.stop_training:
        break

model.save_weights(os.path.join(OUT, 'final.weights.h5'))
